# DATN -- Malaria ProtoCLR Classifier v2
## Single-Phase CE + ProtoCLR Fine-Tune

**Kiến trúc mới (fixed bugs từ v2-orig):**

| Thành phần | Mô tả |
|---|---|
| Phase 1 | CE/Focal + warmup + cosine — backbone trainable, FC head |
| Phase 2 (optional) | ProtoCLR: SupCon(α=0.3) + CE + PushLoss — backbone frozen |
| Calibration | Temperature scaling sau training |
| Evaluate | strict=False khi load checkpoint |

**4 bugs đã fix:**
1. Phase 3 best_state never updated (smooth-history reset inside if-block)
2. calibrated_model.pth save FC keys → evaluate RuntimeError
3. _restore_best_state() overwrite prototypes tốt bằng FC weights
4. evaluate.py strict=True → key mismatch crash

**Dataset:** annotation files tại `BASE_ANN`
**Output:** `/kaggle/working/malaria_proto_v2/`

In [ ]:
!git clone https://github.com/AnhLH04/malaria_clf.git

## 1. Setup & Imports

In [ ]:
import os, sys
MALARIA_SRC = "/kaggle/working/malaria_clf"
if os.path.exists(MALARIA_SRC):
    sys.path.insert(0, MALARIA_SRC)
    print(f"[OK] Source path: {MALARIA_SRC}")
else:
    sys.path.insert(0, "/home/anhlh/Downloads/malaria_proto_clf_src/malaria_clf")
    print("[OK] Using local source path")

from dataset import CLASS_NAMES, NUM_CLASSES, MalariaDataset, get_transforms
from model import MalariaProtoCLFv2, build_model, compute_class_prototypes
from losses import SupConLoss, DynamicFocalLoss
from calibration import TemperatureScaling, compute_ece, PrototypeConfidenceScorer
from train_v2 import TrainerV2SinglePhase, TrainConfigV2
print(f"[OK] Class names: {list(CLASS_NAMES.values())}")
print(f"[OK] NUM_CLASSES: {NUM_CLASSES}")

## 2. Dataset Paths

In [ ]:
BASE_ANN = "/kaggle/input/datasets/khanhtq2101/malaria-parasite/final_malaria_full_class_classification_cropped/5 classes - May 2025"
IMG_BASE = "/kaggle/input/datasets/khanhtq2101/malaria-parasite/final_malaria_full_class_classification_cropped"

for fname in ["train_annotation_5classes.txt", "val_annotation_5classes.txt", "test_annotation_5classes.txt"]:
    path = os.path.join(BASE_ANN, fname)
    if os.path.exists(path):
        with open(path) as f:
            lines = [l for l in f.readlines() if l.strip()]
        print(f"OK {fname}: {len(lines)} samples")
    else:
        print(f"FAIL NOT FOUND: {path}")

# Phan tich class distribution
from collections import Counter
train_labels = []
with open(os.path.join(BASE_ANN, "train_annotation_5classes.txt")) as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.rsplit(None, 1)
            if len(parts) == 2:
                train_labels.append(int(parts[1]))
counts = Counter(train_labels)
print("\nClass distribution (train):")
for cls_id, name in CLASS_NAMES.items():
    print(f"  Class {cls_id} ({name}): {counts.get(cls_id, 0)}")
print(f"  Imbalance ratio (major/minor): {max(counts.values()) / min(counts.values()):.1f}x")

## 3. Training Config

**2-Phase Pipeline:**
- **Phase 1** (30ep): CE/Focal loss, backbone TRAINABLE, FC head
  → Warmup 3ep + CosineAnnealing
  → Focal loss tự động cân bằng class imbalance (Class 4 chiếm 85%)
- **Phase 2** (optional, 15ep): ProtoCLR fine-tune
  → Backbone FROZEN, train prototype + proj_head
  → SupCon(α=0.3) + CE + PushLoss
  → Prototype khởi tạo từ class-mean Phase 1 embeddings

**Bật/tắt ProtoCLR:** `cfg.USE_PROTOCLR = True/False`

In [ ]:
cfg = TrainConfigV2()

# ── Paths ──────────────────────────────────────────────────────────
cfg.BASE_DIR  = BASE_ANN
cfg.TRAIN_ANN = os.path.join(BASE_ANN, "train_annotation_5classes.txt")
cfg.VAL_ANN   = os.path.join(BASE_ANN, "val_annotation_5classes.txt")
cfg.TEST_ANN  = os.path.join(BASE_ANN, "test_annotation_5classes.txt")
cfg.IMG_BASE  = IMG_BASE
cfg.OUTPUT_DIR = "/kaggle/working/malaria_proto_v2"

# ── Model ──────────────────────────────────────────────────────────
cfg.BACKBONE    = "convnextv2_tiny.fcmae_ft_in22k_in1k"
cfg.NUM_CLASSES = 5
cfg.PROJ_DIM    = 128
cfg.IMG_SIZE    = 224
cfg.DROPOUT     = 0.1

# ── Phase 1: CE/Focal baseline ────────────────────────────────────
cfg.EPOCHS          = 30
cfg.BATCH_SIZE      = 32
cfg.LR              = 3e-4
cfg.WEIGHT_DECAY    = 1e-4
cfg.WARMUP_EPOCHS   = 3         # linear warmup before cosine
cfg.LABEL_SMOOTHING = 0.1      # mild label smoothing to reduce overconfidence
cfg.CLF_LOSS        = "focal"  # focal loss: auto-balances minority classes

# ── Phase 2: ProtoCLR fine-tune (optional) ─────────────────────────
cfg.USE_PROTOCLR     = True   # set False to skip ProtoCLR (pure CE baseline)
cfg.PROTOCLR_EPOCHS  = 15
cfg.PROTOCLR_LR_HEAD = 5e-4
cfg.PROTOCLR_LR_BACK = 3e-6   # backbone nearly frozen
cfg.PROTOCLR_ALPHA   = 0.3    # SupCon weight: 30% SupCon, 70% CE
cfg.PUSH_WEIGHT      = 0.1    # prototype push-away loss

# ── Misc ──────────────────────────────────────────────────────────
cfg.SUPCON_TEMP      = 0.07
cfg.EARLY_STOP_PATIENCE  = 8
cfg.EARLY_STOP_MIN_DELTA = 0.002
cfg.DO_CALIBRATION   = True
cfg.SEED             = 42
cfg.NUM_WORKERS      = 4

# ── Print summary ────────────────────────────────────────────────
total = cfg.EPOCHS + (cfg.PROTOCLR_EPOCHS if cfg.USE_PROTOCLR else 0)
print(f"{'='*60}")
print(f"Training config summary:")
print(f"  Backbone:     {cfg.BACKBONE}")
print(f"  Phase 1:      {cfg.EPOCHS}ep  CE/Focal (warmup={cfg.WARMUP_EPOCHS})")
if cfg.USE_PROTOCLR:
    print(f"  Phase 2:      {cfg.PROTOCLR_EPOCHS}ep  ProtoCLR alpha={cfg.PROTOCLR_ALPHA}")
else:
    print(f"  Phase 2:      DISABLED (USE_PROTOCLR=False)")
print(f"  Total epochs: {total}")
print(f"  Clf loss:     {cfg.CLF_LOSS}")
print(f"  Output:       {cfg.OUTPUT_DIR}")
print(f"{'='*60}")

## 4. Run Training

In [ ]:
import warnings
warnings.filterwarnings("ignore")

trainer = TrainerV2SinglePhase(cfg)
model, history = trainer.run()

print(f"\n{'='*60}")
print(f"[Complete] Best macro-F1: {trainer.best_metric:.4f}")
print(f"Checkpoints: {cfg.OUTPUT_DIR}")
print(f"  - phase1_best.pth  - best_protoclr.pth")
print(f"  - calibrated_model.pth  (use this for evaluation)")

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

h = history

# Calculate phase boundaries
p1_end = cfg.EPOCHS
p2_start = p1_end + 1
p2_end   = p1_end + cfg.PROTOCLR_EPOCHS if cfg.USE_PROTOCLR else p1_end

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss
axes[0].plot(h["train_loss"], label="Train Loss", color="blue", linewidth=1.5)
axes[0].plot(h["val_loss"],   label="Val Loss",   color="orange", linewidth=1.5)
if cfg.USE_PROTOCLR:
    axes[0].axvline(p1_end - 1, color="gray", linestyle="--", alpha=0.7, label="P1 -> P2")
axes[0].set_title("Loss Curves"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

# Plot 2: Macro-F1
f1s = h["val_macro_f1"]
axes[1].plot(f1s, label="Val Macro-F1", color="green", linewidth=2)
if cfg.USE_PROTOCLR:
    axes[1].axvline(p1_end - 1, color="gray", linestyle="--", alpha=0.7, label="P1 -> P2")
# Highlight best
best_ep = int(np.argmax(f1s)) + 1
best_f1 = max(f1s)
axes[1].axhline(best_f1, color="red", linestyle=":", alpha=0.6, label=f"Best={best_f1:.4f} (ep{best_ep})")
axes[1].set_title("Validation Macro-F1"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro F1")
axes[1].set_ylim(min(f1s) - 0.02, max(f1s) + 0.02)

plt.suptitle(f"ProtoCLR Training -- {cfg.BACKBONE} | Best F1={best_f1:.4f}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Best val F1: {best_f1:.4f} at epoch {best_ep}")

## 6. Evaluation on Test Set

In [ ]:
from evaluate import evaluate

# Use calibrated_model.pth (best checkpoint after training + temperature scaling)
CHECKPOINT = os.path.join(cfg.OUTPUT_DIR, "calibrated_model.pth")
if not os.path.exists(CHECKPOINT):
    # Fallback: use best_protoclr if ProtoCLR was used
    CHECKPOINT = os.path.join(cfg.OUTPUT_DIR, "best_protoclr.pth")
    print(f"[WARN] calibrated_model.pth not found, using: {CHECKPOINT}")
else:
    print(f"[OK] Using: {CHECKPOINT}")

summary, probs, preds, labels = evaluate(
    checkpoint_path=CHECKPOINT,
    test_ann=cfg.TEST_ANN,
    img_base=cfg.IMG_BASE,
    output_dir=os.path.join(cfg.OUTPUT_DIR, "eval_results"),
    batch_size=64,
)

## 7. Prototype-based Confidence Analysis

**New metrics:**
- `proto_margin`: dist_pred - dist_true (>0 = uncertain)
- `proto_ratio`: dist_pred / (dist_pred + dist_second) (0.5 = equidistant)
- `normalized_entropy`: 1 - H(p)/log(C) (0 = certain, 1 = uncertain)
- `margin_confidence`: P_pred - P_second (softmax-based)

In [ ]:
from misclassification_analysis_v2 import analyze_misclassifications

df_all, df_wrong = analyze_misclassifications(
    checkpoint_path=CHECKPOINT,
    test_ann=cfg.TEST_ANN,
    img_base=cfg.IMG_BASE,
    output_dir=os.path.join(cfg.OUTPUT_DIR, "eval_results"),
    batch_size=64,
    max_per_pair=6,
)
print(f"\n{len(df_wrong)} misclassified out of {len(df_all)}")
print(f"Accuracy: {(df_all['correct'].sum()/len(df_all))*100:.2f}%")

## 8. Proto Confidence Interpretation

In [ ]:
print("INTERPRETATION GUIDE:\n" + "="*60)
print("1. Proto Margin (dist_pred - dist_true)")
print("   < 0 -> embedding near CORRECT class -> confident correct")
print("   ~= 0 -> equidistant -> genuinely ambiguous")
print("   > 0 -> embedding near WRONG class -> uncertain / wrong")
print()
print("2. Proto Ratio = dist_pred / (dist_pred + dist_second)")
print("   < 0.3 -> very confident")
print("   0.3-0.5 -> fairly confident")
print("   0.5 -> equidistant -> NEED ATTENTION")
print("   > 0.7 -> model confused")
print("="*60)

correct_pm = df_all[df_all['correct']]['proto_margin'].mean()
wrong_pm   = df_all[~df_all['correct']]['proto_margin'].mean()
unc_count  = (df_all['proto_ratio'] > 0.6).sum()
print(f"\nKey Stats:")
print(f"  Correct avg proto_margin:  {correct_pm:.4f}")
print(f"  Wrong  avg proto_margin:   {wrong_pm:.4f}")
print(f"  Uncertain (ratio>0.6):    {unc_count} ({unc_count/len(df_all)*100:.1f}%)")
print(f"  Highly uncertain (ratio>0.7): {(df_all['proto_ratio']>0.7).sum()}")

## 9. GradCAM Interpretability Setup

In [ ]:
from gradcam_interpreter import GradCAMInterpreter, PrototypeHeatmapGenerator
import torch
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from train_v2 import load_model_v2
model, temperature = load_model_v2(CHECKPOINT, device)
print(f"Model loaded: {cfg.BACKBONE} | Temperature: {temperature:.4f}")

# ConvNeXtV2: target last stage layer
gradcam = GradCAMInterpreter(model, target_layer_name="backbone.stages.3")
proto_hm_gen = PrototypeHeatmapGenerator(model)
print("GradCAM + ProtoHeatmapGenerator initialized")

## 10. GradCAM -- Demo 1 sample per class

In [ ]:
from torch.utils.data import DataLoader

test_tf  = get_transforms("val", img_size=cfg.IMG_SIZE)
test_ds  = MalariaDataset(cfg.TEST_ANN, cfg.IMG_BASE, transform=test_tf)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

selected = []
seen = set()
for img, label in test_loader:
    if label.item() not in seen:
        seen.add(label.item())
        selected.append((img, label.item()))
        if len(seen) >= NUM_CLASSES:
            break

print(f"Selected {len(selected)} samples (1 per class)")
for img, lbl in selected:
    print(f"  Class {lbl}: {CLASS_NAMES[lbl]}")

In [ ]:
fig, axes = plt.subplots(2, len(selected), figsize=(4*len(selected), 8))

for idx, (img, label) in enumerate(selected):
    img_cuda = img.to(device)
    probs = gradcam.get_prototype_similarities(img_cuda)
    pred  = int(np.argmax(probs))
    overlay = gradcam.generate_overlay(img_cuda, class_idx=pred, alpha=0.4)

    axes[0, idx].imshow(overlay)
    axes[0, idx].set_title(
        f"True: {CLASS_NAMES[label]}\nPred: {CLASS_NAMES[pred]}\nConf: {max(probs):.3f}",
        fontsize=9
    )
    axes[0, idx].axis("off")

    ax = axes[1, idx]
    colors = ["#e74c3c","#3498db","#2ecc71","#f39c12","#95a5a6"]
    bars = ax.bar(list(CLASS_NAMES.values()), probs, color=colors, width=0.6)
    ax.set_ylim(0, 1.1)
    ax.set_title("Proto Similarities", fontsize=9)
    ax.tick_params(axis="x", labelsize=7, rotation=15)
    ax.tick_params(axis="y", labelsize=7)
    for bar, p in zip(bars, probs):
        if p > 0.05:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                    f"{p:.2f}", ha="center", va="bottom", fontsize=7)
    bars[label].set_edgecolor("green"); bars[label].set_linewidth(2)
    bars[pred].set_edgecolor("red");   bars[pred].set_linewidth(2)

plt.suptitle("GradCAM Overlay + Proto Similarities per Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(cfg.OUTPUT_DIR, "eval_results", "gradcam_overview.png"),
    dpi=130, bbox_inches="tight"
)
plt.show()
print("[Saved] gradcam_overview.png")

## 11. Prototype Spatial Similarity Maps

In [ ]:
img, label = selected[1]  # TJ class (index 1)
img_cuda = img.to(device)

pred_idx = int(np.argmax(gradcam.get_prototype_similarities(img_cuda)))
print(f"True: {CLASS_NAMES[label]}, Pred: {CLASS_NAMES[pred_idx]}")
spatial_maps = proto_hm_gen.generate_spatial_maps(img_cuda)

img_np = img.squeeze().cpu().numpy().transpose(1, 2, 0)
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
img_np = np.clip(img_np * std + mean, 0, 1)

fig = plt.figure(figsize=(15, 4))
fig.suptitle(
    f"Prototype Spatial Similarity Maps -- True: {CLASS_NAMES[label]}",
    fontsize=13, fontweight="bold"
)

for idx, (cls_name, sim_map) in enumerate(spatial_maps.items()):
    ax = fig.add_subplot(1, len(spatial_maps), idx + 1)
    ax.imshow(img_np)
    im = ax.imshow(sim_map, cmap="jet", alpha=0.5, vmin=0, vmax=1)
    ax.set_title(f"Proto: {cls_name}", fontsize=10, fontweight="bold")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(
    os.path.join(cfg.OUTPUT_DIR, "eval_results", "proto_spatial_maps.png"),
    dpi=130, bbox_inches="tight"
)
plt.show()
print("[Saved] proto_spatial_maps.png")

## 12. Misclassified Samples -- GradCAM + Proto Metrics

In [ ]:
import pandas as pd

df_wrong = pd.read_csv(os.path.join(cfg.OUTPUT_DIR, "eval_results", "misclassified_v2.csv"))
pair_counts = df_wrong.groupby(["true_label","pred_label"]).size().sort_values(ascending=False)
print("Top confusion pairs:")
print(pair_counts.head(10))
top_pairs = pair_counts.head(2).index.tolist()
print(f"Analyzing: {top_pairs}")

In [ ]:
for true_lbl, pred_lbl in top_pairs[:1]:
    pair_df = df_wrong[(df_wrong["true_label"]==true_lbl) & (df_wrong["pred_label"]==pred_lbl)]
    print(f"\n[{true_lbl} -> {pred_lbl}]: {len(pair_df)} samples")
    print(f"  Avg proto_margin: {pair_df['proto_margin'].mean():.4f}")
    print(f"  Avg proto_ratio:  {pair_df['proto_ratio'].mean():.4f}")
    print(f"  Avg max_conf:     {pair_df['max_conf'].mean():.4f}")

    samples_to_show = pair_df.head(3)
    n = len(samples_to_show)
    if n == 0:
        continue

    fig = plt.figure(figsize=(5*n, 12))
    fig.suptitle(
        f"GradCAM: [{true_lbl} -> {pred_lbl}]\n"
        f"proto_margin={pair_df['proto_margin'].mean():.3f}, "
        f"proto_ratio={pair_df['proto_ratio'].mean():.3f}",
        fontsize=12, fontweight="bold"
    )
    for col_idx, (_, row) in enumerate(samples_to_show.iterrows()):
        try:
            img_pil = Image.open(row["path"]).convert("RGB").resize((cfg.IMG_SIZE, cfg.IMG_SIZE))
            img_tensor = test_tf(img_pil).unsqueeze(0).to(device)
        except:
            print(f"  Cannot load: {row['path']}")
            continue

        overlay = gradcam.generate_overlay(img_tensor, class_idx=row["pred_idx"], alpha=0.4)

        ax_img = fig.add_subplot(4, n, col_idx + 1)
        ax_img.imshow(img_pil)
        ax_img.set_title(
            f"True: {row['true_label']} | Pred: {row['pred_label']}\nconf={row['max_conf']:.3f}",
            fontsize=8
        )
        ax_img.axis("off")

        ax_gc = fig.add_subplot(4, n, n + col_idx + 1)
        ax_gc.imshow(overlay)
        ax_gc.set_title(f"GradCAM (pred={CLASS_NAMES[row['pred_idx']]})", fontsize=8)
        ax_gc.axis("off")

        ax_prob = fig.add_subplot(4, n, 2*n + col_idx + 1)
        probs_row = [row[f"prob_{CLASS_NAMES[i]}"] for i in range(NUM_CLASSES)]
        colors = ["#e74c3c","#3498db","#2ecc71","#f39c12","#95a5a6"]
        bars = ax_prob.bar(list(CLASS_NAMES.values()), probs_row, color=colors, width=0.6)
        ax_prob.set_ylim(0, 1.1)
        ax_prob.tick_params(axis="x", labelsize=6, rotation=15)
        bars[row["true_idx"]].set_edgecolor("green"); bars[row["true_idx"]].set_linewidth(2.5)
        bars[row["pred_idx"]].set_edgecolor("red");   bars[row["pred_idx"]].set_linewidth(2.5)

        ax_txt = fig.add_subplot(4, n, 3*n + col_idx + 1)
        ax_txt.axis("off")
        ax_txt.text(0.1, 0.8, f"proto_margin: {row['proto_margin']:.4f}", fontsize=9, transform=ax_txt.transAxes)
        ax_txt.text(0.1, 0.6, f"proto_ratio:  {row['proto_ratio']:.4f}", fontsize=9, transform=ax_txt.transAxes)
        ax_txt.text(0.1, 0.4, f"margin_conf:  {row['margin_conf']:.4f}", fontsize=9, transform=ax_txt.transAxes)
        ax_txt.text(0.1, 0.2, f"green=true",  fontsize=8, color="green",  transform=ax_txt.transAxes)
        ax_txt.text(0.1, 0.0, f"red=pred",    fontsize=8, color="red",     transform=ax_txt.transAxes)

    plt.tight_layout()
    fname = f"gradcam_analysis_{true_lbl}_to_{pred_lbl}.png"
    plt.savefig(os.path.join(cfg.OUTPUT_DIR, "eval_results", fname), dpi=130, bbox_inches="tight")
    plt.show()
    print(f"[Saved] {fname}")

## 13. Summary Report

In [ ]:
import json

summary_path = os.path.join(cfg.OUTPUT_DIR, "eval_results", "summary_metrics.json")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print("="*60)
    print("EVALUATION SUMMARY")
    print("="*60)
    print(f"  Overall Accuracy:   {summary.get('overall_accuracy', 0)*100:.2f}%")
    print(f"  Macro F1:          {summary.get('macro_f1', 0):.4f}")
    print(f"  Weighted F1:       {summary.get('weighted_f1', 0):.4f}")
    print(f"  ECE:               {summary.get('ece', 0):.4f}")
    print(f"  Parasite Macro F1: {summary.get('parasite_macro_f1', 'N/A')}")
    print(f"  Parasite Weighted F1: {summary.get('parasite_weighted_f1', 'N/A')}")

conf_path = os.path.join(cfg.OUTPUT_DIR, "eval_results", "confidence_summary.json")
if os.path.exists(conf_path):
    with open(conf_path) as f:
        conf = json.load(f)
    print()
    print("PROTOTYPE CONFIDENCE SUMMARY")
    print("="*60)
    print(f"  Correct avg proto_margin:    {conf['correct_proto_margin_mean']:.4f}")
    print(f"  Wrong  avg proto_margin:    {conf['wrong_proto_margin_mean']:.4f}")
    print(f"  Correct avg proto_ratio:    {conf['correct_proto_ratio_mean']:.4f}")
    print(f"  Wrong  avg proto_ratio:     {conf['wrong_proto_ratio_mean']:.4f}")
    print(f"  Uncertain (ratio>0.6):     {conf['uncertain_count']} ({conf['uncertain_count']/conf['total_samples']*100:.1f}%)")
    print(f"  Highly confident (ratio<0.3): {conf['highly_confident']} ({conf['highly_confident']/conf['total_samples']*100:.1f}%)")
else:
    print("confidence_summary.json not found (run Cell 7)")

print()
print("="*60)
print(f"OUTPUT: {os.path.join(cfg.OUTPUT_DIR, 'eval_results')}/")
print("="*60)

## 14. Classification Report

In [ ]:
report_path = os.path.join(cfg.OUTPUT_DIR, "eval_results", "classification_report.txt")
if os.path.exists(report_path):
    with open(report_path) as f:
        content = f.read()
    print("Classification Report:")
    print(content)
else:
    print("Run Cell 6 first to generate classification report")

## 15. Quick Load & Re-evaluate

In [ ]:
# Quick re-evaluate an existing checkpoint without retraining:
# CHECKPOINT = "/kaggle/working/malaria_proto_v2/calibrated_model.pth"
# from train_v2 import load_model_v2
# from misclassification_analysis_v2 import quick_confidence_report
# quick_confidence_report(
#     checkpoint_path=CHECKPOINT,
#     test_ann=cfg.TEST_ANN,
#     img_base=cfg.IMG_BASE,
#     output_dir="/kaggle/working/malaria_proto_v2/eval_results",
# )
print("[INFO] Uncomment cells above to quick-eval an existing checkpoint")